## Just some basic EDA and a how-to guide. ##

In [ ]:
try:
    import cudf as pd

    BACKEND = "cudf (GPU)"
except Exception:  # ImportError, or CUDA/driver errors on GPU-less machines
    import pandas as pd

    BACKEND = "pandas (CPU)"

print(f"Using {BACKEND}")

from pathlib import Path

# Download latest version

# "magic" command to make plots show up in the notebook
%matplotlib inline 
filepath = Path(
    "/home/vlitoux/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3"
)
dataset_path = filepath / "training_data_validated_vs_rejected.parquet"
print("Path to dataset files:", dataset_path)
print("Existing:", dataset_path.exists())

Using cudf (GPU)
Path to dataset files: /home/vlitoux/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3/training_data_validated_vs_rejected.parquet
Existing: True


In [ ]:
entire_df: pd.DataFrame = pd.read_parquet(dataset_path)
entire_df

,is_accepted,loan_amnt,fico_range_low,dti,emp_length,loan_status
0,0,1000.0,693.0,10.00,5.0,Rejected
1,0,1000.0,703.0,10.00,1.0,Rejected
2,0,11000.0,715.0,10.00,2.0,Rejected
3,0,6000.0,698.0,38.64,1.0,Rejected
4,0,1500.0,509.0,9.43,1.0,Rejected
...,...,...,...,...,...,...
2260694,1,24000.0,665.0,19.61,8.0,Current
2260695,1,24000.0,695.0,34.94,11.0,Current
2260696,1,40000.0,705.0,12.75,10.0,Current
2260697,1,24000.0,660.0,18.30,7.0,Charged Off


In [25]:
entire_df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 50000 entries, 399625 to 3812228
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   is_accepted     50000 non-null  int64
 1   loan_amnt       50000 non-null  float64
 2   fico_range_low  50000 non-null  float64
 3   dti             50000 non-null  float64
 4   emp_length      50000 non-null  float64
 5   loan_status     50000 non-null  str
dtypes: float64(4), int64(1), str(1)
memory usage: 2.9 MB


In [ ]:
from cuml import train_test_split
from cuml.preprocessing import StandardScaler

selected_columns = ["loan_amnt", "fico_range_low", "dti", "emp_length"]

if len(entire_df) > 50000:
    entire_df = entire_df.sample(n=50000, random_state=42)

X = entire_df[selected_columns].values
y = entire_df["is_accepted"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.device, X_train_scaled.device

(<CUDA Device 0>, <CUDA Device 0>)

In [ ]:
import torch.nn as nn
import torch


class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x)

class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.linear(x))

In [ ]:
import torch.optim as optim
import torch

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32, device="cuda")
y_train_t = torch.tensor(y_train, dtype=torch.float32, device="cuda").unsqueeze(1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32, device="cuda")
y_test_t = torch.tensor(y_test, dtype=torch.float32, device="cuda")
criterion = nn.BCELoss()

input_dim = X_train_t.shape[1]

loss_table = []
epochs = 500



X_train_t.device, X_test_t.device, y_train_t.device, y_test_t.device


(device(type='cuda', index=0),
 device(type='cuda', index=0),
 device(type='cuda', index=0),
 device(type='cuda', index=0))

In [94]:
X_train_t.min()

tensor(-7.4782, device='cuda:0')

### Logistic regression training

In [91]:
lr = LogisticRegression(input_dim=input_dim)
lr.to("cuda")
optimizer = optim.Adam(lr.parameters(), lr=0.01)
lr.train()

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = lr(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        loss_table.append({"Epoch": epoch + 1, "Loss": loss.item()})

lr.eval()
with torch.no_grad():
    test_probabilities = lr(X_test_t)
    y_pred = (test_probabilities >= 0.5).int().squeeze()
    accuracy = torch.mean((y_pred == y_test_t).float())
print(f"\naccuracy: {accuracy:.4f}")


accuracy: 0.8789


### MLP training

In [ ]:
lr = MLP(input_dim=input_dim)
lr.to("cuda")
optimizer = optim.Adam(lr.parameters(), lr=0.01)
lr.train()

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = lr(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        loss_table.append({"Epoch": epoch + 1, "Loss": loss.item()})

lr.eval()
with torch.no_grad():
    test_probabilities = lr(X_test_t)
    y_pred = (test_probabilities >= 0.5).int().squeeze()
    accuracy = torch.mean((y_pred == y_test_t).float())
print(f"\naccuracy: {accuracy:.4f}")


accuracy: 0.9361
